# Combined MR Summary Analysis

This notebook loads all MR summary CSV files from both b_scans and c_scans,
combines them into a single dataframe, and prepares the data for comprehensive analysis.

## Analysis Components:
1. Load all MR_summary CSV files from b_scans and c_scans
2. Combine into a single dataframe with temperature and scan_type columns
3. Explore the combined dataset
4. Prepare for further analysis (MR vs T, I_min/I_max vs T, etc.)

## 1. Setup & Imports

In [ ]:
# Notebook setup
from scripts.utils import setup_notebook
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

# Additional imports
import glob
import re

## 2. Load All MR Summary CSV Files

In [ ]:
# Define paths to MR_summary directories
b_scans_dir = PROJECT_ROOT / r"output/IV_H_scans/MR_summary/b_scans"
c_scans_dir = PROJECT_ROOT / r"output/IV_H_scans/MR_summary/c_scans"

# Find all CSV files
b_scan_files = list(b_scans_dir.glob("TMR_ratio_vs_V_*.csv"))
c_scan_files = list(c_scans_dir.glob("TMR_ratio_vs_V_*.csv"))

print(f"Found {len(b_scan_files)} b_scan CSV files")
print(f"Found {len(c_scan_files)} c_scan CSV files")
print(f"Total: {len(b_scan_files) + len(c_scan_files)} files")

In [ ]:
def load_mr_csv_with_metadata(csv_path, scan_type):
    """
    Load a single MR summary CSV file and add temperature and scan_type columns.
    
    Parameters:
    -----------
    csv_path : Path
        Path to the CSV file
    scan_type : str
        Either 'b_scan' or 'c_scan'
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with added 'temperature' and 'scan_type' columns
    """
    # Extract temperature from filename (e.g., TMR_ratio_vs_V_10K.csv -> 10)
    filename = csv_path.name
    temp_match = re.search(r'(\d+)K', filename)
    
    if not temp_match:
        print(f"Warning: Could not extract temperature from {filename}")
        return None
    
    temperature = int(temp_match.group(1))
    
    # Load CSV
    df = pd.read_csv(csv_path)
    
    # Add metadata columns
    df['temperature'] = temperature
    df['scan_type'] = scan_type
    
    return df

# Load all b_scan files
b_scan_dataframes = []
for csv_file in b_scan_files:
    df = load_mr_csv_with_metadata(csv_file, 'b_scan')
    if df is not None:
        b_scan_dataframes.append(df)

# Load all c_scan files
c_scan_dataframes = []
for csv_file in c_scan_files:
    df = load_mr_csv_with_metadata(csv_file, 'c_scan')
    if df is not None:
        c_scan_dataframes.append(df)

print(f"\nSuccessfully loaded:")
print(f"  b_scans: {len(b_scan_dataframes)} dataframes")
print(f"  c_scans: {len(c_scan_dataframes)} dataframes")

## 3. Combine All Data into Single DataFrame

In [ ]:
# Combine all dataframes
all_dataframes = b_scan_dataframes + c_scan_dataframes
df_combined = pd.concat(all_dataframes, ignore_index=True)

print(f"\n{'='*70}")
print(f"Combined DataFrame Created!")
print(f"{'='*70}")
print(f"Total rows: {len(df_combined):,}")
print(f"Total columns: {len(df_combined.columns)}")
print(f"\nColumns: {list(df_combined.columns)}")
print(f"\nTemperature range: {df_combined['temperature'].min()}K to {df_combined['temperature'].max()}K")
print(f"Unique temperatures: {sorted(df_combined['temperature'].unique())}")
print(f"\nScan types: {df_combined['scan_type'].unique()}")
print(f"\nData shape: {df_combined.shape}")

In [ ]:
# Display first few rows
print("\nFirst 10 rows of combined data:")
df_combined.head(10)

In [ ]:
# Display summary statistics
print("\nSummary Statistics:")
df_combined.describe()

## 4. Data Quality Check

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df_combined.isnull().sum())
print(f"\nTotal missing values: {df_combined.isnull().sum().sum()}")
print(f"Percentage of missing data: {100 * df_combined.isnull().sum().sum() / (df_combined.shape[0] * df_combined.shape[1]):.2f}%")

In [ ]:
# Count data points per temperature and scan type
print("\nData points per temperature and scan type:")
pivot_counts = df_combined.groupby(['temperature', 'scan_type']).size().unstack(fill_value=0)
print(pivot_counts)

## 5. Save Combined DataFrame

In [ ]:
# Save combined dataframe for future use
output_dir = PROJECT_ROOT / r"output/IV_H_scans/MR_summary"
output_dir.mkdir(parents=True, exist_ok=True)

# Save as pickle for fast loading with data types preserved
pkl_path = output_dir / "MR_summary_combined_all_temps.pkl"
df_combined.to_pickle(pkl_path)
print(f"Saved combined dataframe (pickle): {pkl_path}")

# Save as CSV for portability
csv_path = output_dir / "MR_summary_combined_all_temps.csv"
df_combined.to_csv(csv_path, index=False)
print(f"Saved combined dataframe (CSV): {csv_path}")

## 6. Quick Visualization: MR vs Voltage for All Temperatures

In [ ]:
import matplotlib.cm as cm

# ============================================================
# FILTER CONFIGURATION - Adjust these values as needed
# ============================================================

# Minimum I_apar threshold (A) - removes noisy low-current data
i_apar_threshold = 2e-9  # Remove data where |I_apar| < 1 nA

# Voltage cutoffs per temperature (K: max_voltage in V)
# Only temperatures listed here will have voltage limits applied
# Leave empty {} to show full voltage range for all temperatures
voltage_cutoffs = {
  
    110: 0.5,
    120: 0.5,
    130:0.5,
    140:0.5,
    150:0.5,
    160:0.5

    # Example configurations (uncomment and adjust as needed):
    # 50: 0.25,   # Limit 50K data to |V| <= 0.25V
    # 60: 0.25,   # Limit 60K data to |V| <= 0.25V
    # 100: 0.5,   # Limit 100K data to |V| <= 0.5V
}

# Plot style configuration
plot_with_errorbars = True  # Set to False to plot without error bars (faster)
errorbar_alpha = 0.7  # Transparency of error bars (0-1)

# ============================================================


def apply_filters(data, temp, i_threshold, v_cutoffs):
    """
    Apply current and voltage filters to TMR data.
    
    Parameters:
    -----------
    data : pd.DataFrame
        DataFrame subset for a specific temperature/scan_type
    temp : int or float
        Temperature value
    i_threshold : float
        Minimum |I_apar| threshold in Amperes
    v_cutoffs : dict
        Dictionary of {temperature: max_voltage} for voltage cutoffs
    
    Returns:
    --------
    pd.DataFrame
        Filtered DataFrame
    """
    # Filter 1: Remove data where |I_apar| < threshold
    filtered = data[np.abs(data['I_apar (A)']) >= i_threshold].copy()
    
    # Filter 2: Apply voltage cutoff if specified for this temperature
    if temp in v_cutoffs:
        max_v = v_cutoffs[temp]
        filtered = filtered[np.abs(filtered['Voltage (V)']) <= max_v]
    
    return filtered


# Create figure with two subplots (one for each scan type)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Get unique temperatures and create colormap
temperatures = [20,30,40,50,60,70]

# Use tab20 colormap for better contrast between adjacent temperatures
# Other good options: 'tab20b', 'tab20c', 'turbo', 'twilight', 'hsv'
cmap = cm.get_cmap('tab20', len(temperatures))

# Plot b_scans
for i, temp in enumerate(temperatures):
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'b_scan')]
    data = apply_filters(data, temp, i_apar_threshold, voltage_cutoffs)
    if len(data) > 0:
        color = cmap(i)
        if plot_with_errorbars:
            ax1.errorbar(data['Voltage (V)'], data['TMR_Ratio']-1, 
                        yerr=data['TMR_Error'],
                        fmt='o', color=color, linewidth=1.5, 
                        ecolor=color, elinewidth=0.5, capsize=0,
                        errorevery=5, alpha=errorbar_alpha,
                        label=f'{temp}K')
        else:
            ax1.plot(data['Voltage (V)'], data['TMR_Ratio'], '-', 
                    color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')

ax1.set_xlabel('Voltage (V)', fontsize=12)
ax1.set_ylabel('MR Ratio', fontsize=12)
ax1.set_title('MR vs Voltage - b_scans', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0,4)
ax1.legend()

# Plot c_scans
for i, temp in enumerate(temperatures):
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'c_scan')]
    data = apply_filters(data, temp, i_apar_threshold, voltage_cutoffs)
    if len(data) > 0:
        color = cmap(i)
        if plot_with_errorbars:
            ax2.errorbar(data['Voltage (V)'], data['TMR_Ratio']-1, 
                        yerr=data['TMR_Error'],
                        fmt='o', color=color, linewidth=1.5,
                        ecolor=color, elinewidth=0.5, capsize=0,
                        errorevery=5, alpha=errorbar_alpha,
                        label=f'{temp}K')
        else:
            ax2.plot(data['Voltage (V)'], data['TMR_Ratio'], '-', 
                    color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')

ax2.set_xlabel('Voltage (V)', fontsize=12)
ax2.set_ylabel('MR Ratio', fontsize=12)
ax2.set_title('MR vs Voltage - c_scans', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0,4)
ax2.legend()


plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.cm as cm

# ============================================================
# FILTER CONFIGURATION - Adjust these values as needed
# ============================================================

# Minimum I_apar threshold (A) - removes noisy low-current data
i_apar_threshold = 2e-9  # Remove data where |I_apar| < 1 nA

# Voltage cutoffs per temperature (K: max_voltage in V)
# Only temperatures listed here will have voltage limits applied
# Leave empty {} to show full voltage range for all temperatures
voltage_cutoffs = {
  
    110: 0.5,
    120: 0.5,
    130:0.5,
    140:0.5,
    150:0.5,
    160:0.5

    # Example configurations (uncomment and adjust as needed):
    # 50: 0.25,   # Limit 50K data to |V| <= 0.25V
    # 60: 0.25,   # Limit 60K data to |V| <= 0.25V
    # 100: 0.5,   # Limit 100K data to |V| <= 0.5V
}

# Plot style configuration
plot_with_errorbars = True  # Set to False to plot without error bars (faster)
errorbar_alpha = 0.6  # Transparency of error bars (0-1)

# ============================================================


def apply_filters(data, temp, i_threshold, v_cutoffs):
    """
    Apply current and voltage filters to TMR data.
    
    Parameters:
    -----------
    data : pd.DataFrame
        DataFrame subset for a specific temperature/scan_type
    temp : int or float
        Temperature value
    i_threshold : float
        Minimum |I_apar| threshold in Amperes
    v_cutoffs : dict
        Dictionary of {temperature: max_voltage} for voltage cutoffs
    
    Returns:
    --------
    pd.DataFrame
        Filtered DataFrame
    """
    # Filter 1: Remove data where |I_apar| < threshold
    filtered = data[np.abs(data['I_apar (A)']) >= i_threshold].copy()
    
    # Filter 2: Apply voltage cutoff if specified for this temperature
    if temp in v_cutoffs:
        max_v = v_cutoffs[temp]
        filtered = filtered[np.abs(filtered['Voltage (V)']) <= max_v]
    
    return filtered


# Create figure with two subplots (one for each scan type)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Get unique temperatures and create colormap
temperatures = [80,90,100,110,120,130,140,150,160]

wrong_temp_array = [50, 60, 70, 80, 90]
fix_multiplier = 1

# Use tab20 colormap for better contrast between adjacent temperatures
# Other good options: 'tab20b', 'tab20c', 'turbo', 'twilight', 'hsv'
cmap = cm.get_cmap('tab20', len(temperatures))

# Plot b_scans
for i, temp in enumerate(temperatures):
    fix_multiplier = -1 if temp in wrong_temp_array else 1
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'b_scan')]
    data = apply_filters(data, temp, i_apar_threshold, voltage_cutoffs)
    if len(data) > 0:
        color = cmap(i)
        if plot_with_errorbars:
            ax1.errorbar(fix_multiplier*data['Voltage (V)'], data['TMR_Ratio']-1, 
                        yerr=data['TMR_Error'],
                        fmt='o', color=color, linewidth=1.5, 
                        ecolor=color, elinewidth=0.5, capsize=0,
                        errorevery=5, alpha=errorbar_alpha,
                        label=f'{temp}K')
        else:
            ax1.plot(data['Voltage (V)'], data['TMR_Ratio'], '-', 
                    color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')

ax1.set_xlabel('Voltage (V)', fontsize=12)
ax1.set_ylabel('MR Ratio', fontsize=12)
ax1.set_title('MR vs Voltage - b_scans', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0,4)
ax1.legend()

# Plot c_scans
for i, temp in enumerate(temperatures):
    fix_multiplier = -1 if temp in wrong_temp_array else 1
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'c_scan')]
    data = apply_filters(data, temp, i_apar_threshold, voltage_cutoffs)
    if len(data) > 0:
        color = cmap(i)
        if plot_with_errorbars:
            ax2.errorbar(fix_multiplier*data['Voltage (V)'], data['TMR_Ratio']-1, 
                        yerr=data['TMR_Error'],
                        fmt='o', color=color, linewidth=1.5,
                        ecolor=color, elinewidth=0.5, capsize=0,
                        errorevery=5, alpha=errorbar_alpha,
                        label=f'{temp}K')
        else:
            ax2.plot(data['Voltage (V)'], data['TMR_Ratio'], '-', 
                    color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')

ax2.set_xlabel('Voltage (V)', fontsize=12)
ax2.set_ylabel('MR Ratio', fontsize=12)
ax2.set_title('MR vs Voltage - c_scans', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0,4)

ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.cm as cm

# Create figure with two subplots (one for each scan type)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Get unique temperatures and create colormap
temperatures = sorted(df_combined['temperature'].unique())
cmap = cm.get_cmap('viridis')
norm = plt.Normalize(vmin=min(temperatures), vmax=max(temperatures))

# Plot b_scans
for temp in temperatures:
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'b_scan')]
    if len(data) > 0:
        color = cmap(norm(temp))
        ax1.plot(data['Voltage (V)'], data['I_par (A)'], '-', 
                color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')

ax1.set_xlabel('Voltage (V)', fontsize=12)
ax1.set_ylabel('I_par (A)', fontsize=12)
ax1.set_title('I_par vs Voltage - b_scans', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Plot c_scans
for temp in temperatures:
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'c_scan')]
    if len(data) > 0:
        color = cmap(norm(temp))
        ax2.plot(data['Voltage (V)'], data['I_par (A)'], '-', 
                color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')

ax2.set_xlabel('Voltage (V)', fontsize=12)
ax2.set_ylabel('I_par (A)', fontsize=12)
ax2.set_title('I_par vs Voltage - c_scans', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=[ax1, ax2], orientation='vertical', pad=0.02)
cbar.set_label('Temperature (K)', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.cm as cm

# Create figure with two subplots (one for each scan type)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Get unique temperatures and create colormap
temperatures = [40, 50, 60, 70, 80, 90, 100]
cmap = cm.get_cmap('viridis')
norm = plt.Normalize(vmin=min(temperatures), vmax=max(temperatures))

# Plot b_scans
for temp in temperatures:
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'b_scan')]
    if len(data) > 0:
        color = cmap(norm(temp))
        ax1.plot(data['Voltage (V)'], data['I_par (A)'], '-', 
                color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')

ax1.set_xlabel('Voltage (V)', fontsize=12)
ax1.set_ylabel('I_par (A)', fontsize=12)
ax1.set_title('I_par vs Voltage - b_scans', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')
ax1.set_xscale('log')


# Plot c_scans
for temp in temperatures:
    data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == 'c_scan')]
    if len(data) > 0:
        color = cmap(norm(temp))
        ax2.plot(data['Voltage (V)'], data['I_par (A)'], '-', 
                color=color, linewidth=1.5, alpha=0.7, label=f'{temp}K')

ax2.set_xlabel('Voltage (V)', fontsize=12)
ax2.set_ylabel('I_par (A)', fontsize=12)
ax2.set_title('I_par vs Voltage - c_scans', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)


# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, ax=[ax1, ax2], orientation='vertical', pad=0.02)
cbar.set_label('Temperature (K)', fontsize=12)

plt.tight_layout()
plt.show()

## 9. I_par at Fixed Voltage vs Temperature

Analyze how the parallel current (I_par) varies with temperature at a fixed bias voltage of 0.5V.

In [ ]:
# Extract I_par at V=0.25V for each temperature and scan type
target_voltage = 0.1  # V
tolerance = 0.01  # Voltage tolerance for matching

def get_current_at_voltage(group, v_target, tolerance=0.01):
    """Extract current closest to target voltage."""
    voltage_diff = np.abs(group['Voltage (V)'] - v_target)
    closest_idx = voltage_diff.idxmin()
    
    if voltage_diff.loc[closest_idx] < tolerance:
        return pd.Series({
            'I_par': group.loc[closest_idx, 'I_par (A)'],
            'I_par_error': group.loc[closest_idx, 'I_par_error (A)'],
            'voltage_actual': group.loc[closest_idx, 'Voltage (V)'],
            'I_apar': group.loc[closest_idx, 'I_apar (A)'],
            'I_apar_error': group.loc[closest_idx, 'I_apar_error (A)']
        })
    else:
        return pd.Series({
            'I_par': np.nan,
            'I_par_error': np.nan,
            'voltage_actual': np.nan,
            'I_apar': np.nan,
            'I_apar_error': np.nan
        })

# Apply to each temperature/scan_type group
i_par_vs_temp = df_combined.groupby(['temperature', 'scan_type']).apply(
    lambda g: get_current_at_voltage(g, target_voltage, tolerance)
).reset_index()

# Separate b_scan and c_scan
b_scan_data = i_par_vs_temp[i_par_vs_temp['scan_type'] == 'b_scan'].sort_values('temperature')
c_scan_data = i_par_vs_temp[i_par_vs_temp['scan_type'] == 'c_scan'].sort_values('temperature')

print(f"I_par at {target_voltage}V vs Temperature")
print(f"\nb_scans:")
print(b_scan_data)
print(f"\nc_scans:")
print(c_scan_data)

In [ ]:
# Plot I_par at 0.5V vs Temperature for b and c-axis
fig, ax = plt.subplots(figsize=(12, 7))

# Plot b_scans
ax.errorbar(1/b_scan_data['temperature'], 1/b_scan_data['I_par'],
            yerr=(1/b_scan_data['I_par_error'])/1000,
            fmt='o-', color='blue', markersize=8, linewidth=2,
            capsize=5, capthick=2, label='b-axis', alpha=0.8)

# Plot c_scans
ax.errorbar(1/c_scan_data['temperature'], 1/c_scan_data['I_par'],
            yerr=1/c_scan_data['I_par_error'],
            fmt='s-', color='red', markersize=8, linewidth=2,
            capsize=5, capthick=2, label='c-axis', alpha=0.8)

ax.set_xlabel('1/Temperature (K-1)', fontsize=14)
ax.set_ylabel('1/I_par (A-1)', fontsize=14)
ax.set_title(f'Parallel Current (I_par) at {target_voltage}V vs Temperature', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=12, loc='best')
ax.set_yscale('log')
ax.set_xlim(0,0.05)

plt.tight_layout()
plt.show()

In [ ]:
# Plot I_par at 0.5V vs Temperature for b and c-axis
fig, ax = plt.subplots(figsize=(12, 7))

# Plot b_scans
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_par'],
            yerr=(b_scan_data['I_par_error']),
            fmt='o-', color='blue', markersize=8, linewidth=2,
            capsize=5, capthick=2, label='b-axis', alpha=0.8)

# Plot c_scans
ax.errorbar(c_scan_data['temperature'], c_scan_data['I_par'],
            yerr=c_scan_data['I_par_error'],
            fmt='s-', color='red', markersize=8, linewidth=2,
            capsize=5, capthick=2, label='c-axis', alpha=0.8)

ax.set_xlabel('1/Temperature (K-1)', fontsize=14)
ax.set_ylabel('1/I_par (A-1)', fontsize=14)
ax.set_title(f'Parallel Current (I_par) at {target_voltage}V vs Temperature', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=12, loc='best')
ax.set_yscale('log')


plt.tight_layout()
plt.show()

In [ ]:
# Plot I_apar and I_par at 0.25V vs Temperature with dual y-axes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left subplot: b-axis
ax1_right = ax1.twinx()  # Create right y-axis for I_par

# Plot I_apar on left axis (convert to µA)
line1 = ax1.errorbar(b_scan_data['temperature'], b_scan_data['I_apar'] * 1e6,
                     yerr=b_scan_data['I_apar_error'] * 1e6,
                     fmt='o-', color='blue', markersize=8, linewidth=2,
                     capsize=5, capthick=2, label='I_apar', alpha=0.8)

# Plot I_par on right axis (keep in A, convert to µA for display)
line2 = ax1_right.errorbar(b_scan_data['temperature'], b_scan_data['I_par'] * 1e6,
                            yerr=b_scan_data['I_par_error'] * 1e6,
                            fmt='s-', color='darkblue', markersize=8, linewidth=2,
                            capsize=5, capthick=2, label='I_par', alpha=0.8)

ax1.set_xlabel('Temperature (K)', fontsize=14)
ax1.set_ylabel('I_apar (µA)', fontsize=14, color='blue')
ax1_right.set_ylabel('I_par (µA)', fontsize=14, color='darkblue')
ax1.set_title(f'b-axis: I_apar and I_par at {target_voltage}V vs Temperature', 
              fontsize=14, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='blue')
ax1_right.tick_params(axis='y', labelcolor='darkblue')
ax1.grid(True, alpha=0.3, linestyle='--')

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_right.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=12, loc='upper left')

# Right subplot: c-axis
ax2_right = ax2.twinx()  # Create right y-axis for I_par

# Plot I_apar on left axis (convert to µA)
line3 = ax2.errorbar(c_scan_data['temperature'], c_scan_data['I_apar'] * 1e6,
                     yerr=c_scan_data['I_apar_error'] * 1e6,
                     fmt='o-', color='red', markersize=8, linewidth=2,
                     capsize=5, capthick=2, label='I_apar', alpha=0.8)

# Plot I_par on right axis (convert to µA for display)
line4 = ax2_right.errorbar(c_scan_data['temperature'], c_scan_data['I_par'] * 1e6,
                            yerr=c_scan_data['I_par_error'] * 1e6,
                            fmt='s-', color='darkred', markersize=8, linewidth=2,
                            capsize=5, capthick=2, label='I_par', alpha=0.8)

ax2.set_xlabel('Temperature (K)', fontsize=14)
ax2.set_ylabel('I_apar (µA)', fontsize=14, color='red')
ax2_right.set_ylabel('I_par (µA)', fontsize=14, color='darkred')
ax2.set_title(f'c-axis: I_apar and I_par at {target_voltage}V vs Temperature', 
              fontsize=14, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='red')
ax2_right.tick_params(axis='y', labelcolor='darkred')
ax2.grid(True, alpha=0.3, linestyle='--')

# Combine legends
lines3, labels3 = ax2.get_legend_handles_labels()
lines4, labels4 = ax2_right.get_legend_handles_labels()
ax2.legend(lines3 + lines4, labels3 + labels4, fontsize=12, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# Compare I_par at 0.25V: b-axis vs c-axis
fig, ax = plt.subplots(figsize=(12, 7))

# Plot b-axis I_par
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_par'] * 1e6,
            yerr=b_scan_data['I_par_error'] * 1e6,
            fmt='o-', color='blue', markersize=8, linewidth=2,
            capsize=5, capthick=2, label='b-axis', alpha=0.8)

# Plot c-axis I_par
ax.errorbar(c_scan_data['temperature'], c_scan_data['I_par'] * 1e6,
            yerr=c_scan_data['I_par_error'] * 1e6,
            fmt='s-', color='red', markersize=8, linewidth=2,
            capsize=5, capthick=2, label='c-axis', alpha=0.8)

ax.set_xlabel('Temperature (K)', fontsize=14)
ax.set_ylabel('I_par (µA)', fontsize=14)
ax.set_title(f'Parallel Current (I_par) at {target_voltage}V vs Temperature: b-axis vs c-axis', 
             fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=12, loc='best') # Set lower limit for log scale
plt.tight_layout()
plt.show()

In [ ]:
# Compare I_apar at 0.25V: b-axis vs c-axis
fig, ax = plt.subplots(figsize=(12, 7))

# Plot b-axis I_apar
ax.errorbar(b_scan_data['temperature'], b_scan_data['I_apar'] * 1e6,
            yerr=b_scan_data['I_apar_error'] * 1e6,
            fmt='o-', color='blue', markersize=8, linewidth=2,
            capsize=5, capthick=2, label='b-axis', alpha=0.8)

# Plot c-axis I_apar
ax.errorbar(c_scan_data['temperature'], c_scan_data['I_apar'] * 1e6,
            yerr=c_scan_data['I_apar_error'] * 1e6,
            fmt='s-', color='red', markersize=8, linewidth=2,
            capsize=5, capthick=2, label='c-axis', alpha=0.8)

ax.set_xlabel('Temperature (K)', fontsize=14)
ax.set_ylabel('I_apar (µA)', fontsize=14)
ax.set_title(f'Anti-parallel Current (I_apar) at {target_voltage}V vs Temperature: b-axis vs c-axis', 
             fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=12, loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate and plot voltage at peak TMR vs temperature
import pandas as pd
import matplotlib.pyplot as plt

peak_tmr_data = []

# Assuming df_combined and apply_filters are in the namespace
if 'df_combined' in locals() or 'df_combined' in globals():
    temperatures = df_combined['temperature'].unique()
    temperatures.sort()

    for temp in temperatures:
        for scan_type in ['b_scan', 'c_scan']:
            data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == scan_type)]
            if len(data) == 0:
                continue
            
            # Use 'i_apar_threshold' and 'voltage_cutoffs' if defined
            try:
                filtered_data = apply_filters(data, temp, i_apar_threshold, voltage_cutoffs)
            except NameError:
                # Fallback if filters are not available in the current namespace
                filtered_data = data
                
            if len(filtered_data) > 0:
                # Find the row with maximum TMR_Ratio
                max_tmr_idx = filtered_data['TMR_Ratio'].idxmax()
                peak_voltage = filtered_data.loc[max_tmr_idx, 'Voltage (V)']
                max_tmr = filtered_data.loc[max_tmr_idx, 'TMR_Ratio']
                peak_tmr_data.append({
                    'temperature': temp,
                    'scan_type': scan_type,
                    'peak_voltage': peak_voltage,
                    'max_tmr': max_tmr
                })

    if peak_tmr_data:
        df_peak_tmr = pd.DataFrame(peak_tmr_data)

        # Create the plot
        fig, ax = plt.subplots(figsize=(10, 6))

        # Plot b_scan
        b_scan_peaks = df_peak_tmr[df_peak_tmr['scan_type'] == 'b_scan']
        if not b_scan_peaks.empty:
            ax.plot(b_scan_peaks['temperature'], b_scan_peaks['peak_voltage'], 'o-', label='b_scan', color='blue', linewidth=2, markersize=8)

        # Plot c_scan
        c_scan_peaks = df_peak_tmr[df_peak_tmr['scan_type'] == 'c_scan']
        if not c_scan_peaks.empty:
            ax.plot(c_scan_peaks['temperature'], c_scan_peaks['peak_voltage'], 's-', label='c_scan', color='red', linewidth=2, markersize=8)

        ax.set_xlabel('Temperature (K)', fontsize=12)
        ax.set_ylabel('Voltage at Peak MR (V)', fontsize=12)
        ax.set_title('Voltage at Peak MR vs Temperature', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No peak TMR data found after filtering.")
else:
    print("df_combined not found in namespace.")


In [ ]:
# Calculate and plot both positive and negative side voltage peaks at peak TMR vs temperature
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

peak_data_pos = []
peak_data_neg = []

if 'df_combined' in locals() or 'df_combined' in globals():
    temperatures = df_combined['temperature'].unique()
    temperatures.sort()

    for temp in temperatures:
        for scan_type in ['b_scan', 'c_scan']:
            data = df_combined[(df_combined['temperature'] == temp) & (df_combined['scan_type'] == scan_type)]
            if len(data) == 0:
                continue
            
            try:
                filtered_data = apply_filters(data, temp, i_apar_threshold, voltage_cutoffs)
            except NameError:
                filtered_data = data
                
            # Split data into positive and negative voltage regions
            data_pos = filtered_data[filtered_data['Voltage (V)'] > 0]
            data_neg = filtered_data[filtered_data['Voltage (V)'] < 0]
            
            # Positive side peak
            if len(data_pos) > 0:
                max_tmr_idx_pos = data_pos['TMR_Ratio'].idxmax()
                peak_voltage_pos = data_pos.loc[max_tmr_idx_pos, 'Voltage (V)']
                max_tmr_pos = data_pos.loc[max_tmr_idx_pos, 'TMR_Ratio']
                peak_data_pos.append({
                    'temperature': temp,
                    'scan_type': scan_type,
                    'peak_voltage': peak_voltage_pos,
                    'max_tmr': max_tmr_pos
                })
                
            # Negative side peak
            if len(data_neg) > 0:
                max_tmr_idx_neg = data_neg['TMR_Ratio'].idxmax()
                peak_voltage_neg = data_neg.loc[max_tmr_idx_neg, 'Voltage (V)']
                max_tmr_neg = data_neg.loc[max_tmr_idx_neg, 'TMR_Ratio']
                peak_data_neg.append({
                    'temperature': temp,
                    'scan_type': scan_type,
                    'peak_voltage': peak_voltage_neg,
                    'max_tmr': max_tmr_neg
                })

    # Create the plot
    fig, ax = plt.subplots(figsize=(10, 6))

    if peak_data_pos:
        df_pos = pd.DataFrame(peak_data_pos)
        b_scan_pos = df_pos[df_pos['scan_type'] == 'b_scan']
        c_scan_pos = df_pos[df_pos['scan_type'] == 'c_scan']
        
        if not b_scan_pos.empty:
            ax.plot(b_scan_pos['temperature'], b_scan_pos['peak_voltage'], 'o-', label='b_scan (V > 0)', color='blue', linewidth=2, markersize=8)
        if not c_scan_pos.empty:
            ax.plot(c_scan_pos['temperature'], c_scan_pos['peak_voltage'], 's-', label='c_scan (V > 0)', color='red', linewidth=2, markersize=8)

    if peak_data_neg:
        df_neg = pd.DataFrame(peak_data_neg)
        b_scan_neg = df_neg[df_neg['scan_type'] == 'b_scan']
        c_scan_neg = df_neg[df_neg['scan_type'] == 'c_scan']
        
        if not b_scan_neg.empty:
            ax.plot(b_scan_neg['temperature'], b_scan_neg['peak_voltage'], 'o--', label='b_scan (V < 0)', color='lightblue', linewidth=2, markersize=8)
        if not c_scan_neg.empty:
            ax.plot(c_scan_neg['temperature'], c_scan_neg['peak_voltage'], 's--', label='c_scan (V < 0)', color='lightcoral', linewidth=2, markersize=8)

    ax.set_xlabel('Temperature (K)', fontsize=12)
    ax.set_ylabel('Voltage at Peak MR (V)', fontsize=12)
    ax.set_title('Positive and Negative Side Voltage Peaks vs Temperature', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Place legend outside to avoid obscuring data
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.xlim(15,165)
    plt.ylim(-0.6,0.6)
    plt.tight_layout()
    plt.show()
else:
    print("df_combined not found in namespace.")


In [ ]:
# Plot maximum TMR for positive bias voltage and maximum TMR for negative bias voltage
if peak_data_pos or peak_data_neg:
    fig, ax = plt.subplots(figsize=(10, 6))

    if peak_data_pos:
        df_pos = pd.DataFrame(peak_data_pos)
        b_scan_pos = df_pos[df_pos['scan_type'] == 'b_scan']
        c_scan_pos = df_pos[df_pos['scan_type'] == 'c_scan']
        
        if not b_scan_pos.empty:
            ax.plot(b_scan_pos['temperature'], b_scan_pos['max_tmr'], 'o-', label='b_scan (V > 0)', color='blue', linewidth=2, markersize=8)
        if not c_scan_pos.empty:
            ax.plot(c_scan_pos['temperature'], c_scan_pos['max_tmr'], 's-', label='c_scan (V > 0)', color='red', linewidth=2, markersize=8)

    if peak_data_neg:
        df_neg = pd.DataFrame(peak_data_neg)
        b_scan_neg = df_neg[df_neg['scan_type'] == 'b_scan']
        c_scan_neg = df_neg[df_neg['scan_type'] == 'c_scan']
        
        if not b_scan_neg.empty:
            ax.plot(b_scan_neg['temperature'], b_scan_neg['max_tmr'], 'o--', label='b_scan (V < 0)', color='lightblue', linewidth=2, markersize=8)
        if not c_scan_neg.empty:
            ax.plot(c_scan_neg['temperature'], c_scan_neg['max_tmr'], 's--', label='c_scan (V < 0)', color='lightcoral', linewidth=2, markersize=8)

    ax.set_xlabel('Temperature (K)', fontsize=12)
    ax.set_ylabel('Maximum MR Ratio', fontsize=12)

    ax.set_xlim(15, 165)
    ax.set_ylim(0, 5)
    ax.set_title('Maximum MR Ratio (Positive and Negative Bias) vs Temperature', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Place legend outside to avoid obscuring data
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()

else:
    print("Required data not found. Please run the previous cell first.")
